# Travel Booking Analysis (2023-2025)

## Project Overview

This project explores travel booking data from 2023 to 2025 to understand booking trends, business growth, and travel package performance.

Using Python, Pandas, and NumPy, I cleaned and combined booking records, analyzed performance across three years, and examined how booking value was distributed across different travel package categories.

The analysis focuses on three questions:

1. How did booking activity change between 2023 and 2025?
2. Which travel packages generated the highest booking value?
3. Which package categories contributed the most to total booking value?

## Objectives

- Analyze booking trends and business performance across three years.
- Identify the highest performing travel packages.
- Examine booking value across package categories.
- Apply data cleaning, transformation, aggregation, and merging techniques.
- Use NumPy to reshape analysis results and export the final output to CSV.

## Tools and Technologies

- **Python:** Data processing and analysis
- **Pandas:** Data cleaning, transformation, aggregation, and merging
- **NumPy:** Array operations and data reshaping
- **Jupyter Notebook:** Analysis and documentation

## 1. Data Loading and Setup

The analysis uses booking records from 2023, 2024, and 2025. I loaded the three datasets and combined them into a single DataFrame to analyze booking activity across the full period.

In [1]:
# Importing libraries

import numpy as np
import pandas as pd


In [2]:
# load the booking datasets
# Load annual booking datasets
df2023 = pd.read_csv("../data/raw/Bookings_2023 (1).csv")
df2024 = pd.read_csv("../data/raw/Bookings_2024 (1).csv")
df2025 = pd.read_csv("../data/raw/Bookings_2025 (1).csv")

# Combine booking records
df = pd.concat([df2023, df2024, df2025], ignore_index=True)

# Check the combined dataset
df.shape

(13927, 12)

## 2. Data Exploration

Before cleaning the data, I examined the dataset's structure, data types, missing values, and duplicate records to identify issues that could affect the analysis.

In [3]:
# Check dataset structure and data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13927 entries, 0 to 13926
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   BookingID        13927 non-null  str    
 1   BookingDate      13927 non-null  str    
 2   TravelStartDate  13927 non-null  str    
 3   CustomerID       13927 non-null  str    
 4   PackageID        13927 non-null  str    
 5   AgentID          12827 non-null  str    
 6   Adults           13927 non-null  int64  
 7   Children         13927 non-null  int64  
 8   UnitPriceUSD     13927 non-null  int64  
 9   DiscountPct      13510 non-null  float64
 10  Status           13927 non-null  str    
 11  PaymentMethod    13927 non-null  str    
dtypes: float64(1), int64(3), str(8)
memory usage: 2.1 MB


In [4]:
# Check missing values
df.isna().sum()

BookingID             0
BookingDate           0
TravelStartDate       0
CustomerID            0
PackageID             0
AgentID            1100
Adults                0
Children              0
UnitPriceUSD          0
DiscountPct         417
Status                0
PaymentMethod         0
dtype: int64

In [5]:
# Check duplicate records
df.duplicated().sum()

np.int64(37)

## 3. Data Cleaning

The initial checks identified duplicate bookings, missing values, and date columns stored as text. I cleaned the data to prepare it for the business analysis.

In [6]:
# Remove duplicate booking records
df = df.drop_duplicates(subset="BookingID").reset_index(drop=True)

# Verify the result
print("Remaining bookings:", len(df))
print("Duplicate booking IDs:", df["BookingID"].duplicated().sum())

Remaining bookings: 13890
Duplicate booking IDs: 0


In [7]:
# Clean missing values
df["AgentID"] = df["AgentID"].fillna("Unknown")
df["DiscountPct"] = df["DiscountPct"].fillna(0)

# Verify missing values after cleaning.
df[["AgentID", "DiscountPct"]].isna().sum()

AgentID        0
DiscountPct    0
dtype: int64

Missing AgentID values were replaced with "Unknown", while missing DiscountPct values were filled with 0.

In [8]:
# Standardize booking status labels
df["Status"] = (
    df["Status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({"canceled": "cancelled"})
)

# Check the cleaned status values
df["Status"].value_counts(dropna=False)

Status
completed    12408
cancelled     1312
confirmed      170
Name: count, dtype: int64[pyarrow]

In [9]:
# Convert booking dates to datetime
df["BookingDate"] = pd.to_datetime(
    df["BookingDate"],
    errors="coerce"
)

# Extract booking year and month
df["BookingYear"] = df["BookingDate"].dt.year
df["BookingMonth"] = df["BookingDate"].dt.month

# Check whether any dates failed to convert
print("Invalid booking dates:", df["BookingDate"].isna().sum())

Invalid booking dates: 0


## 4. Business Growth Analysis

This section examines booking activity and total booking value between 2023 and 2025. I compared the number of bookings, travellers served, and booking value across the three years.

In [10]:
# Calculate travellers per booking
df["Travellers"] = df["Adults"] + df["Children"]

# Calculate booking values
df["GrossValue"] = df["UnitPriceUSD"]
df["DiscountAmount"] = df["GrossValue"] * df["DiscountPct"]
df["GrossNet"] = df["GrossValue"] - df["DiscountAmount"]

# Preview calculated columns
df[
    ["Travellers", "GrossValue", "DiscountAmount", "GrossNet"]
].head()

,Travellers,GrossValue,DiscountAmount,GrossNet
0,2,1250,0.0,1250.0
1,1,2990,0.0,2990.0
2,5,4485,0.0,4485.0
3,1,1250,0.0,1250.0
4,1,420,63.0,357.0


In [11]:
# Business
# 1. Is our business growing?
# Year over year analysis
# Metrics to use --> Travellers growth/clients number growth
# Financial Growth over three years
# Number of bookings across the three years to asses growth
BusinessGrowth = (
    df.groupby("BookingYear")
    .agg(
        TotalBookingValue=("GrossValue", "sum"),
        TravellersServed=("Travellers", "sum"),
        Bookings_made=("BookingID", "nunique")
    )
    .reset_index()
)

BusinessGrowth

,BookingYear,TotalBookingValue,TravellersServed,Bookings_made
0,2023,10511475,13540,3867
1,2024,14265495,15905,4589
2,2025,19011415,18955,5434


## 5. Package Performance Analysis

I compared travel packages by total booking value and number of bookings to identify the best-performing packages.

In [12]:
# 2. which packages perform best
PackagePerformance = (
    df.groupby("PackageID")
    .agg(
        TotalBookingValue=("GrossValue", "sum"),
        Bookings_made=("BookingID", "nunique")
    )
    .reset_index()
    .sort_values("TotalBookingValue", ascending=False)
)

PackagePerformance.head()

,PackageID,TotalBookingValue,Bookings_made
11,PK12,6253025,1101
12,PK13,6085400,828
16,PK17,5057095,435
15,PK16,3529125,647
13,PK14,3334230,686


In [13]:
# Load travel package details
packages = pd.read_csv("../data/raw/Packages (1).csv")

# Check the package dataset
packages.shape

(18, 8)

In [14]:
# Merge package details with booking performance
PackagePerformanceDetails = PackagePerformance.merge(
    packages,
    on="PackageID",
    how="left"
)

# Display the top three packages
PackagePerformanceDetails[
    ["PackageID", "PackageName", "TotalBookingValue", "Bookings_made"]
].head(3)

,PackageID,PackageName,TotalBookingValue,Bookings_made
0,PK12,Mara Great Migration Luxury Camp,6253025,1101
1,PK13,Serengeti Migration Fly-In,6085400,828
2,PK17,Ultimate East Africa Honeymoon,5057095,435


## 6. Booking Value by Category

I examined how booking value was distributed across travel package categories between 2023 and 2025.

To do this, I combined the booking records with package information and summarized booking value, booking counts, and travellers by year and category.

In [15]:
# 3  Where does booking value come from?
# Add package categories to the booking records
df = df.merge(
    packages[["PackageID", "Category"]],
    on="PackageID",
    how="left",
    validate="many_to_one"
)

# Summarize booking value by year and category
BookingValueSource = (
    df.groupby(["BookingYear", "Category"])
    .agg(
        TotalBookingValue=("GrossValue", "sum"),
        Bookings_made=("BookingID", "nunique"),
        TravellersServed=("Travellers", "sum")
    )
    .reset_index()
)

BookingValueSource

,BookingYear,Category,TotalBookingValue,Bookings_made,TravellersServed
0,2023,Budget,718035,1290,4514
1,2023,Luxury,6639420,1106,3912
2,2023,Mid-Range,3154020,1471,5114
3,2024,Budget,806335,1387,4732
4,2024,Luxury,9861160,1583,5530
5,2024,Mid-Range,3598000,1619,5643
6,2025,Budget,945975,1556,5359
7,2025,Luxury,14115070,2188,7713
8,2025,Mid-Range,3950370,1690,5883


In [16]:
# Get the unique years and travel categories
years = BookingValueSource["BookingYear"].unique()
categories = BookingValueSource["Category"].unique()

# Convert booking values into a NumPy array
booking_values = BookingValueSource["TotalBookingValue"].to_numpy()

# Reshape the values into rows (years) and columns (categories)
booking_values_reshaped = booking_values.reshape(
    len(years), len(categories)
)

# Create a table showing booking value by year and category
BookingValuePivot = pd.DataFrame(
    booking_values_reshaped,
    index=years,
    columns=categories
)

BookingValuePivot.index.name = "BookingYear"

BookingValuePivot

,Budget,Luxury,Mid-Range
BookingYear,,,
2023,718035,6639420,3154020
2024,806335,9861160,3598000
2025,945975,14115070,3950370


### Findings: Where Does Booking Value Come From?

The analysis shows that the **Luxury** category generated the highest booking value in all three years, followed by Mid-Range and Budget.

Luxury booking value increased from $6,639,420 in 2023 to $9,861,160 in 2024, reaching $14,115,070 in 2025.

This indicates that Luxury travel packages were the largest contributors to total booking value between 2023 and 2025.

In [17]:
# 4. Export the data into a CSV
BookingValuePivot.to_csv(
    "../outputs/booking_value_analysis.csv",
    index_label="BookingYear"
)